# Session 3: simulate the removed material and check it
**A. Real-world problem.** A toolpath that looks right can still leave material behind,
cut too much, or take twice as long as it should. We simulate the cutter on a grid of
heights (a *height-map*) and compare the removed volume and the leftover ridges with formulas.

**Plan.** (1) a height-map and a cutter stamp, (2) raster and contour-parallel paths,
(3) scallop height for a ball-end cutter, (4) MRR and chip load, (5) corners, gouges and
cutters that do not fit, (6) a pocket with a round boss, from G-code to volume check.

In [ ]:
import math
import pathlib
import re

import numpy as np

# where figures are saved (works as a script and in Jupyter/Colab)
OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"

## Step 1: a height-map and a cutter stamp
The stock is a grid of cells; each stores the height of the material top (0 = original top,
-5 = 5 mm down). A flat-end cutter at (x, y) with tip height z sets every cell within its
radius to at most z. A ball-end cutter of radius r leaves a rounded surface:
height = z + r - sqrt(r^2 - rho^2) at distance rho from the axis.

In [ ]:
class HeightMap:
    def __init__(self, x0, x1, y0, y1, res):
        self.x0, self.y0, self.res = x0, y0, res
        self.nx, self.ny = int(round((x1 - x0) / res)), int(round((y1 - y0) / res))
        self.xc = x0 + (np.arange(self.nx) + 0.5) * res      # cell centres
        self.yc = y0 + (np.arange(self.ny) + 0.5) * res
        self.h = np.zeros((self.ny, self.nx))

    def stamp(self, x, y, z, r, ball=False):
        res = self.res
        i0, i1 = max(0, int((x - r - self.x0) / res) - 1), min(self.nx, int((x + r - self.x0) / res) + 2)
        j0, j1 = max(0, int((y - r - self.y0) / res) - 1), min(self.ny, int((y + r - self.y0) / res) + 2)
        if i0 >= i1 or j0 >= j1:
            return
        d2 = (self.xc[None, i0:i1] - x) ** 2 + (self.yc[j0:j1, None] - y) ** 2
        inside = d2 <= r * r
        surf = z + r - np.sqrt(np.maximum(r * r - d2, 0.0)) if ball else np.full(d2.shape, float(z))
        view = self.h[j0:j1, i0:i1]
        np.minimum(view, np.where(inside, surf, np.inf), out=view)

    def cut_polyline(self, pts, z, r, ball=False):
        """Sweep the cutter along a polyline at constant tip height z."""
        pts = np.asarray(pts, dtype=float)
        for a, b in zip(pts[:-1], pts[1:]):
            n = max(1, int(np.ceil(np.linalg.norm(b - a) / self.res)))
            for t in np.linspace(0.0, 1.0, n + 1):
                self.stamp(*(a + t * (b - a)), z, r, ball)

    def removed_volume(self):
        """Volume of removed material in mm^3."""
        return float(-self.h.sum() * self.res ** 2)

    def removed_area(self, floor=-1e-9):
        """Footprint area (mm^2) of cells cut at all."""
        return float((self.h < floor).sum() * self.res ** 2)

## Step 2: two pocketing strategies
*Raster (zigzag)*: parallel lines spaced by the stepover s. *Contour-parallel*: the pocket
outline offset inwards by r, then by r + s, r + 2s, ... until nothing is left (rings, like a
spiral). Both keep the cutter centre inside the (w - D) x (h - D) rectangle.

In [ ]:
def n_lines(span, s):
    """Lines needed to cover a span with spacing at most s (end lines included)."""
    return math.ceil(span / s - 1e-9) + 1


def raster_path(w, h, d, s):
    """Zigzag centre path for a w x h rectangular pocket and cutter diameter d."""
    r, n = d / 2, n_lines(h - d, s)
    ys = np.linspace(r, h - r, n)
    pts = []
    for j, y in enumerate(ys):
        a, b = (r, w - r) if j % 2 == 0 else (w - r, r)
        pts += [(a, y), (b, y)]
    return np.array(pts)


def contour_path(w, h, d, s):
    """Rings of inward rectangle offsets (first ring at the cutter radius), joined into one path."""
    r, pts, k = d / 2, [], 0
    while True:
        x0, y0, x1, y1 = r + k * s, r + k * s, w - r - k * s, h - r - k * s
        if x1 - x0 < -1e-9 or y1 - y0 < -1e-9:
            break
        if y1 - y0 < 1e-9:
            pts += [(x0, y0), (x1, y0)]
        elif x1 - x0 < 1e-9:
            pts += [(x0, y0), (x0, y1)]
        else:
            pts += [(x0, y0), (x1, y0), (x1, y1), (x0, y1), (x0, y0)]
        k += 1
    return np.array(pts)


def path_length(pts):
    return float(np.hypot(*np.diff(np.asarray(pts, dtype=float), axis=0).T).sum())

## Step 3: ridges (scallops) left by a ball-end cutter
Neighbouring passes s apart leave a ridge midway. The ball surface at horizontal distance
s/2 from its axis is r - sqrt(r^2 - (s/2)^2) above the tip, so the ridge height is
h = r - sqrt(r^2 - (s/2)^2). Inverse: s = 2 sqrt(2 r h - h^2).

In [ ]:
def cusp_height(r, s):
    half = min(s / 2.0, r)
    return r - math.sqrt(r * r - half * half)


def stepover_for_cusp(r, h):
    return 2.0 * math.sqrt(2.0 * r * h - h * h)


def simulate_cusp(r, s, n_pass=5, res=0.025, length=12.0):
    """Measured ridge height between the middle passes of n_pass parallel ball-end passes."""
    ymid = (n_pass - 1) * s / 2.0
    hm = HeightMap(0.0, length, -res / 2 - 0.5 * s, (n_pass - 1) * s + 0.5 * s, res)
    tip = -1.0
    for k in range(n_pass):
        hm.cut_polyline([(-r, k * s), (length + r, k * s)], tip, r, ball=True)
    rows = (hm.yc > ymid + 1e-9) & (hm.yc < ymid + s - 1e-9)      # between passes 3 and 4
    cols = (hm.xc > r) & (hm.xc < length - r)
    return float(hm.h[np.ix_(rows, cols)].max() - tip)

## Step 4: material removal rate and chip load
MRR = width of cut * depth of cut * feed (mm^3/min). Chip load (feed per tooth)
f_z = F / (N z) with spindle speed N (rev/min) and z teeth.

In [ ]:
def mrr(ae, ap, feed):
    """Material removal rate, mm^3/min (ae = radial width, ap = axial depth, mm)."""
    return ae * ap * feed


def chip_load(feed, rpm, teeth):
    """Feed per tooth in mm."""
    return feed / (rpm * teeth)


def line_cut_volume(r, ap, length, res=0.05):
    """Simulated volume of a single straight flat-end cut of the given length."""
    hm = HeightMap(-r - 1, length + r + 1, -r - 1, r + 1, res)
    hm.cut_polyline([(0.0, 0.0), (length, 0.0)], -ap, r)
    return hm.removed_volume()

## Step 5: corners, leftovers, wrong offsets and cutters that do not fit
A round cutter cannot reach into a sharp inside corner: each corner keeps
r^2 (1 - pi/4) of leftover area. A stepover larger than the cutter diameter leaves strips.
An offset in the wrong direction gouges the walls. A slot narrower than the cutter cannot
be cut at all; the planner must say so.

In [ ]:
def corner_leftover(r):
    """Total leftover area (mm^2) in the four inside corners of a rectangular pocket."""
    return 4 * r * r * (1 - math.pi / 4)


def pocket_cleared_area(path, w, h, r, res=0.1, margin=0.0):
    """Simulate a flat cutter along path; return (cleared area, area cut outside the pocket)."""
    hm = HeightMap(-margin, w + margin, -margin, h + margin, res)
    hm.cut_polyline(path, -1.0, r)
    cut = hm.h < -1e-9
    inside = (hm.xc[None, :] >= 0) & (hm.xc[None, :] <= w) & (hm.yc[:, None] >= 0) & (hm.yc[:, None] <= h)
    return float((cut & inside).sum() * res ** 2), float((cut & ~inside).sum() * res ** 2)


def wrong_offset_gouge_area(w, h, r):
    """Analytic area cut outside the pocket when the centre path is OFFSET OUTWARDS by r."""
    return (w + 4 * r) * (h + 4 * r) - (4 - math.pi) * r * r - w * h


def plan_slot(length, width, d):
    """Centre line of a straight slot, or a clear message if the cutter is too big."""
    if d > width + 1e-12:
        return None, f"cutter diameter {d:g} mm is wider than the {width:g} mm slot: cannot cut"
    r = d / 2
    return [(r, width / 2), (length - r, width / 2)], "ok"


def check_inside_corner(corner_radius, d):
    """An inside corner cannot be sharper than the cutter radius."""
    if corner_radius + 1e-12 < d / 2:
        return f"corner radius {corner_radius:g} mm is smaller than the cutter radius {d / 2:g} mm: leftover remains"
    return "ok"

## Step 6: G-code for a pocket with a round boss, then simulate that G-code
Rows are split where the cutter centre would enter a circle of radius (boss radius + r); the
cutter lifts over the boss. A full circle G3 trims around it and a wall lap finishes the outline. We parse the text back
and cut the height-map from the parsed moves.

In [ ]:
def boss_pocket_gcode(w, h, depth, d, doc, s, feed, plunge, boss_c, boss_d, lift=1.0):
    r, R = d / 2, boss_d / 2 + d / 2
    layers = math.ceil(depth / doc - 1e-9)
    dz = depth / layers
    n = n_lines(h - d, s)
    ys = np.linspace(r, h - r, n)
    f = lambda v: f"{v:.3f}".rstrip("0").rstrip(".")
    out = ["G21", "G90", f"G0 Z{f(lift)}"]
    for k in range(1, layers + 1):
        z = -k * dz
        for j, y in enumerate(ys):
            dy = y - boss_c[1]
            if abs(dy) < R - 1e-9:
                half = math.sqrt(R * R - dy * dy)
                segs = [(r, boss_c[0] - half), (boss_c[0] + half, w - r)]
            else:
                segs = [(r, w - r)]
            if j % 2 == 1:
                segs = [(b, a) for a, b in reversed(segs)]
            for a, b in segs:
                out += [f"G0 X{f(a)} Y{f(y)}", f"G1 Z{f(z)} F{f(plunge)}", f"G1 X{f(b)} F{f(feed)}", f"G0 Z{f(lift)}"]
        out += [f"G0 X{f(boss_c[0] + R)} Y{f(boss_c[1])}", f"G1 Z{f(z)} F{f(plunge)}",
                f"G3 X{f(boss_c[0] + R)} Y{f(boss_c[1])} I{f(-R)} J0 F{f(feed)}", f"G0 Z{f(lift)}"]
        # wall pass: one lap round the pocket outline, which removes the scallops at the line ends
        out += [f"G0 X{f(r)} Y{f(r)}", f"G1 Z{f(z)} F{f(plunge)}", f"G1 X{f(w - r)} F{f(feed)}",
                f"G1 Y{f(h - r)}", f"G1 X{f(r)}", f"G1 Y{f(r)}", f"G0 Z{f(lift)}"]
    out.append("G0 Z5")
    return "\n".join(out) + "\n"


TOKEN = re.compile(r"([GXYZIJF])(-?[0-9]*\.?[0-9]+)")


def read_moves(text):
    """Parse G0/G1/G3 text (modal) into dicts with kind, start, end, centre offsets, feed."""
    x = y = z = 0.0
    kind, feed, moves = "G0", 0.0, []
    for line in text.splitlines():
        t = {k: float(v) for k, v in TOKEN.findall(line.split("(")[0].upper())}
        if "G" in t and int(t["G"]) in (0, 1, 2, 3):
            kind = f"G{int(t['G'])}"
        feed = t.get("F", feed)
        if not ({"X", "Y", "Z"} & set(t)):
            continue
        nx_, ny_, nz_ = t.get("X", x), t.get("Y", y), t.get("Z", z)
        moves.append({"kind": kind, "a": (x, y, z), "b": (nx_, ny_, nz_), "ij": (t.get("I", 0.0), t.get("J", 0.0)), "f": feed})
        x, y, z = nx_, ny_, nz_
    return moves


def move_points(m, step):
    """Sample points (x, y, z) along a parsed move."""
    a, b = np.array(m["a"]), np.array(m["b"])
    if m["kind"] in ("G2", "G3"):
        c = a[:2] + np.array(m["ij"])
        rad = float(np.hypot(*(a[:2] - c)))
        a0, a1 = math.atan2(*(a[1::-1] - c[1::-1])), math.atan2(*(b[1::-1] - c[1::-1]))
        sw = (a1 - a0) % (2 * math.pi) if m["kind"] == "G3" else (a0 - a1) % (2 * math.pi)
        if sw < 1e-12 and np.allclose(a[:2], b[:2]):
            sw = 2 * math.pi
        n = max(1, int(math.ceil(rad * sw / step)))
        sgn = 1.0 if m["kind"] == "G3" else -1.0
        u = np.linspace(0.0, 1.0, n + 1)
        ang = a0 + sgn * sw * u
        return np.stack([c[0] + rad * np.cos(ang), c[1] + rad * np.sin(ang), a[2] + (b[2] - a[2]) * u], axis=1)
    n = max(1, int(math.ceil(np.linalg.norm(b - a) / step)))
    u = np.linspace(0.0, 1.0, n + 1)[:, None]
    return a + u * (b - a)


def moves_length(moves, kinds=("G1", "G2", "G3")):
    total = 0.0
    for m in moves:
        if m["kind"] in kinds:
            pts = move_points(m, 0.02)
            total += float(np.linalg.norm(np.diff(pts, axis=0), axis=1).sum())
    return total


def gcode_time_s(moves, rapid_rate=3000.0):
    t = 0.0
    for m in moves:
        rate = rapid_rate if m["kind"] == "G0" else m["f"]
        pts = move_points(m, 0.02)
        t += float(np.linalg.norm(np.diff(pts, axis=0), axis=1).sum()) / rate * 60.0
    return t


def simulate_moves(hm, moves, r):
    """Cut the height-map with every cutting move of a parsed program."""
    for m in moves:
        if m["kind"] == "G0":
            continue
        for x, y, z in move_points(m, hm.res):
            hm.stamp(x, y, z, r)
    return hm


def boss_expected_volume(w, h, depth, d, boss_d):
    """Analytic target: pocket minus boss minus four corner leftovers, times depth (mm^3)."""
    return (w * h - math.pi * (boss_d / 2) ** 2 - corner_leftover(d / 2)) * depth

## Step 7: see it

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    W, H, D = 40.0, 30.0, 6.0
    for name, path in (("raster s=4", raster_path(W, H, D, 4)), ("contour s=4", contour_path(W, H, D, 4))):
        cl, out = pocket_cleared_area(path, W, H, D / 2)
        print(f"{name}: length {path_length(path):.1f} mm, cleared {cl:.1f} of {W * H:g} mm^2")
    for s in (1, 2, 3, 4):
        print(f"ball r=3, s={s}: formula {cusp_height(3, s):.4f} mm, simulated {simulate_cusp(3, s):.4f} mm")
    prog = boss_pocket_gcode(W, H, 5, D, 2, 4, 300, 100, (20, 15), 12)
    mv = read_moves(prog)
    hm = HeightMap(0, W, 0, H, 0.1)
    simulate_moves(hm, mv, D / 2)
    exp = boss_expected_volume(W, H, 5, D, 12)
    print(f"boss pocket: simulated {hm.removed_volume():.0f} mm^3, expected {exp:.0f} mm^3, time {gcode_time_s(mv):.0f} s")

    fig, axs = plt.subplots(1, 3, figsize=(13, 4.4))
    ax = axs[0]
    im = ax.imshow(hm.h, origin="lower", extent=(0, W, 0, H), cmap="viridis")
    fig.colorbar(im, ax=ax, label="material height (mm)", fraction=0.046)
    ax.set_xlabel("x (mm)")
    ax.set_ylabel("y (mm)")
    ax.set_title("Simulated pocket with a 12 mm boss")
    ss = np.linspace(1.0, 6.0, 51)
    ax = axs[1]
    ax.plot(ss, [path_length(raster_path(W, H, D, s)) / 300 * 60 for s in ss], label="raster")
    ax.plot(ss, [path_length(contour_path(W, H, D, s)) / 300 * 60 for s in ss], label="contour-parallel")
    ax.set_xlabel("stepover (mm)")
    ax.set_ylabel("one-layer cutting time (s) at 300 mm/min")
    ax.set_title("Bigger stepover, shorter time")
    ax.legend()
    ax.grid(alpha=0.3)
    ax = axs[2]
    rs = np.linspace(0.5, 5, 40)
    ax.plot(rs, [corner_leftover(r) for r in rs], label="formula 4 r^2 (1 - pi/4)")
    ax.set_xlabel("cutter radius (mm)")
    ax.set_ylabel("leftover in four corners (mm^2)")
    ax.set_title("Inside corners: bigger cutter, more left")
    ax.legend()
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(OUT / "boss_and_tradeoffs.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6, 4))
    ss = np.linspace(0.2, 4.0, 60)
    ax2.plot(ss, [cusp_height(3, s) for s in ss], label="formula, r = 3 mm")
    ax2.plot([1, 2, 3, 4], [simulate_cusp(3, s) for s in (1, 2, 3, 4)], "o", label="height-map simulation")
    ax2.set_xlabel("stepover (mm)")
    ax2.set_ylabel("ridge height (mm)")
    ax2.set_title("Ball-end ridge height against stepover")
    ax2.legend()
    ax2.grid(alpha=0.3)
    fig2.tight_layout()
    fig2.savefig(OUT / "cusp_height.png", dpi=120)
    print("saved figures in", OUT)

## Try it
Do the coding tasks, answer the four questions in the README (section F), then run the
checkpoint.

## Coding tasks
1. **Easy.** Use `simulate_cusp(3, 1.5)` and compare it with `cusp_height(3, 1.5)`; then try a ball radius of 5 mm.
2. **Stretch.** Change `boss_pocket_gcode` so that the stepover is 3 mm and print the time and the removed volume check.
3. **Explain.** Explain why the planner must refuse a 6 mm cutter in a 5 mm slot instead of silently cutting a 6 mm slot.

## Assessment
- **A. Conceptual.** Why does a bigger stepover finish a job faster but leave more material behind? Why does the contour-parallel path with stepover equal to the diameter still leave cusps in the corners?
- **B. Numerical.** A cutter has 2 teeth, runs at 8000 rev/min and feeds at 480 mm/min. Find the chip load. Then find the MRR for a_e = 3 mm, a_p = 1.5 mm and F = 400 mm/min.
- **C. Graph and simulation interpretation.** In the corner-leftover graph, read the leftover area for a cutter radius of 3 mm and of 5 mm. What happens when the radius doubles, and what does that say about choosing a cutter?
- **D. Design and optimisation challenge.** Design: a ball-end cutter of radius 4 mm must finish a 40 x 30 mm face with a ridge of at most 0.02 mm at 600 mm/min. Choose a stepover (rounded down to 0.01 mm), count the lines and estimate the time. Lines run along the 40 mm side, plus 30 mm of side-steps in total.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import math

import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    W, H, D = 40.0, 30.0, 6.0
    # --- scallops (ball end)
    for s, want in ((1, 0.0420), (2, 0.1716), (3, 0.4019), (4, 0.7639)):
        checks.append((f"ridge height r=3 mm, stepover {s} mm: formula {want:.4f} mm",
                       abs(L.cusp_height(3, s) - want) < 5e-5))
    checks.append(("worked example: the height-map simulation agrees with the formula to 0.002 mm (stepover 1 to 4 mm)",
                   all(abs(L.simulate_cusp(3, s) - L.cusp_height(3, s)) < 2e-3 for s in (1, 2, 3, 4))))
    checks.append(("inverse formula: ridge 0.05 mm with r = 3 mm needs stepover 1.0909 mm",
                   abs(L.stepover_for_cusp(3, 0.05) - 1.0909) < 1e-4 and abs(L.cusp_height(3, L.stepover_for_cusp(3, 0.05)) - 0.05) < 1e-12))
    checks.append(("stepover 1.0 mm gives 0.0420 mm (<= 0.05) but 1.1 mm gives 0.0508 mm (> 0.05)",
                   L.cusp_height(3, 1.0) <= 0.05 < L.cusp_height(3, 1.1)))
    checks.append(("edge: stepover equal to the cutter diameter gives a finite ridge of r = 3 mm; zero stepover gives 0",
                   abs(L.cusp_height(3, 6) - 3.0) < 1e-12 and L.cusp_height(3, 0) == 0.0 and L.stepover_for_cusp(3, 0) == 0.0))
    # --- MRR and chip load
    checks.append(("MRR = 4 mm * 2 mm * 300 mm/min = 2400 mm^3/min", L.mrr(4, 2, 300) == 2400))
    checks.append(("chip load = 300 / (6000 * 2) = 0.025 mm per tooth", abs(L.chip_load(300, 6000, 2) - 0.025) < 1e-12))
    v = L.line_cut_volume(3, 2, 20)
    cap = math.pi * 9 * 2
    checks.append(("a 20 mm straight cut (6 mm wide, 2 mm deep): simulated volume = (6*20 + pi*9) * 2 = 296.5 mm^3 within 0.5%",
                   abs(v - (6 * 20 * 2 + cap)) / v < 0.005))
    checks.append(("MRR check: removed volume minus the two half-circle ends equals width*depth*length = 240 mm^3 (feed cancels: 6*2*300 * 20/300)",
                   abs((v - cap) - L.mrr(6, 2, 300) * 20 / 300) / 240 < 0.005))
    # --- strategies
    ras, con = L.raster_path(W, H, D, 4), L.contour_path(W, H, D, 4)
    checks.append(("raster with stepover 4: 7 lines, length 262 mm per layer", L.n_lines(24, 4) == 7 and abs(L.path_length(ras) - 262) < 1e-9))
    checks.append(("contour-parallel with stepover 4: rings 34x24, 26x16, 18x8 and a 10 mm centre line, length 279.0 mm",
                   abs(L.path_length(con) - 278.97) < 0.01 and len(con) == 5 * 3 + 2))
    cl_c, out_c = L.pocket_cleared_area(con, W, H, 3)
    ideal = W * H - L.corner_leftover(3)
    checks.append(("contour-parallel clears the pocket except the corners: 1192.3 mm^2 (= 1200 - 4 * 3^2 * (1 - pi/4))",
                   abs(cl_c - ideal) < 0.2 and abs(ideal - 1192.27) < 0.01 and out_c == 0.0))
    cl_r, _ = L.pocket_cleared_area(ras, W, H, 3)
    checks.append(("raster alone leaves extra scallops at the line ends (cleared 1186.5 mm^2 < 1192.3)",
                   cl_r < cl_c - 5.0 and abs(cl_r - 1186.5) < 0.2))
    checks.append(("time/stepover: raster lengths 466, 262, 194 mm for stepover 2, 4, 6 mm",
                   [round(L.path_length(L.raster_path(W, H, D, s))) for s in (2, 4, 6)] == [466, 262, 194]))
    checks.append(("a stepover of 2r leaves corner cusps in contour-parallel: cleared 1180.9 mm^2 at s = 6 mm",
                   abs(L.pocket_cleared_area(L.contour_path(W, H, D, 6), W, H, 3)[0] - 1180.9) < 0.2))
    # --- strips (stepover larger than the diameter)
    wide = L.raster_path(W, H, D, 8)
    hm = L.HeightMap(0, W, 0, H, 0.1)
    hm.cut_polyline(wide, -1.0, 3.0)
    win = (hm.xc >= 6) & (hm.xc <= 34)
    uncut = float((hm.h[:, win] >= -1e-9).sum() * 0.01)
    checks.append(("stepover 8 mm with a 6 mm cutter leaves 3 strips 2 mm wide: 3 * 2 * 28 = 168 mm^2 uncut between x = 6 and 34",
                   L.n_lines(24, 8) == 4 and abs(uncut - 168.0) < 1.0))
    # --- corners
    ok = True
    for r in (1, 2, 3, 4, 5):
        pts = np.vstack([L.raster_path(W, H, 2 * r, r), [(r, r), (W - r, r), (W - r, H - r), (r, H - r), (r, r)]])
        cleared, _ = L.pocket_cleared_area(pts, W, H, r, res=0.05)
        ok &= abs((W * H - cleared) - L.corner_leftover(r)) < 0.1
    checks.append(("inside corners: leftover area 4 r^2 (1 - pi/4) matches the simulation for r = 1..5 mm (0.86, 3.43, 7.73, 13.73, 21.46)", ok))
    checks.append(("corner leftover formula values for r = 1 and 3 mm", abs(L.corner_leftover(1) - 0.8584) < 1e-4 and abs(L.corner_leftover(3) - 7.7257) < 1e-4))
    checks.append(("planner: corner radius 2 mm with a 6 mm cutter is reported; 3 mm is fine",
                   "smaller" in L.check_inside_corner(2, 6) and L.check_inside_corner(3, 6) == "ok"))
    # --- failures
    inside = np.array([(3, 3), (37, 3), (37, 27), (3, 27), (3, 3)], dtype=float)
    outward = np.array([(-3, -3), (43, -3), (43, 33), (-3, 33), (-3, -3)], dtype=float)
    on_line = np.array([(0, 0), (40, 0), (40, 30), (0, 30), (0, 0)], dtype=float)
    gi = L.pocket_cleared_area(inside, W, H, 3, res=0.05, margin=10)
    go = L.pocket_cleared_area(outward, W, H, 3, res=0.05, margin=10)
    gl = L.pocket_cleared_area(on_line, W, H, 3, res=0.05, margin=10)
    checks.append(("correct inside offset gouges nothing (0 mm^2 outside the pocket)", gi[1] == 0.0))
    checks.append(("wrong direction (outward offset): cuts 976.3 mm^2 outside the pocket and nothing inside",
                   abs(go[1] - L.wrong_offset_gouge_area(W, H, 3)) < 1.0 and abs(L.wrong_offset_gouge_area(W, H, 3) - 976.27) < 0.01 and go[0] == 0.0))
    checks.append(("zero offset (centre on the outline) gouges 448.3 mm^2 outside the pocket", abs(gl[1] - 448.26) < 1.0))
    checks.append(("slot 5 mm wide, 6 mm cutter: the planner reports it cannot cut",
                   L.plan_slot(30, 5, 6)[0] is None and "cannot cut" in L.plan_slot(30, 5, 6)[1]))
    checks.append(("edge: a 6 mm cutter in a 6 mm slot is accepted, centre line from x = 3 to 27 mm",
                   L.plan_slot(30, 6, 6)[1] == "ok" and L.plan_slot(30, 6, 6)[0] == [(3.0, 3.0), (27.0, 3.0)]))
    # --- mini-project
    prog = L.boss_pocket_gcode(W, H, 5, D, 2, 4, 300, 100, (20, 15), 12)
    mv = L.read_moves(prog)
    checks.append(("boss program is plain G-code: G21, G90, uses G0, G1 and G3 only",
                   prog.startswith("G21\nG90") and {m["kind"] for m in mv} == {"G0", "G1", "G3"}))
    sim = L.HeightMap(0, W, 0, H, 0.1)
    L.simulate_moves(sim, mv, D / 2)
    expected = L.boss_expected_volume(W, H, 5, D, 12)
    checks.append(("worked example: expected removed volume (1200 - 113.10 - 7.73) * 5 = 5395.9 mm^3", abs(expected - 5395.9) < 0.1))
    checks.append(("simulation of the parsed G-code removes 5396 mm^3, within 0.1% of the expected volume",
                   abs(sim.removed_volume() - expected) / expected < 0.001))
    cx, cy = np.argmin(abs(sim.xc - 20)), np.argmin(abs(sim.yc - 15))
    checks.append(("the boss is untouched (height 0) and the floor is at -5 mm", sim.h[cy, cx] == 0.0 and abs(sim.h.min() + 5.0) < 1e-9))
    checks.append(("the uncut area is the boss plus four corners (about 120.8 mm^2)",
                   abs(float((sim.h > -1e-9).sum() * 0.01) - 120.76) < 0.5))
    t = L.gcode_time_s(mv)
    checks.append(("estimated time: 327 s (5.5 min) with rapids at 3000 mm/min", abs(t - 327.3) < 0.5))
    checks.append(("the program has 1213 mm of cutting, arc and plunge moves", abs(L.moves_length(mv) - 1213.4) < 0.5))
    checks.append(("corner leftover for r = 2, 4, 5 mm: 3.43, 13.73, 21.46 mm^2; doubling r quadruples it",
                   abs(L.corner_leftover(2) - 3.43) < 0.01 and abs(L.corner_leftover(4) - 13.73) < 0.01 and abs(L.corner_leftover(5) - 21.46) < 0.01
                   and abs(L.corner_leftover(4) / L.corner_leftover(2) - 4.0) < 1e-12))
    checks.append(("chip load 480 / (8000 * 2) = 0.03 mm and MRR 3 * 1.5 * 400 = 1800 mm^3/min",
                   abs(L.chip_load(480, 8000, 2) - 0.03) < 1e-12 and abs(L.mrr(3, 1.5, 400) - 1800) < 1e-9))
    checks.append(("worked example arithmetic: sqrt(8) = 2.828, sqrt(0.2975) = 0.5454, ridge at 1.1 mm = 0.0508 mm",
                   abs(math.sqrt(8) - 2.828) < 5e-4 and abs(math.sqrt(0.2975) - 0.5454) < 5e-5 and abs(L.cusp_height(3, 1.1) - 0.0508) < 5e-5))
    s_d = L.stepover_for_cusp(4, 0.02)
    checks.append(("design answer: ball r = 4 mm, ridge 0.02 mm: s = 0.7990 mm, use 0.79 mm: 39 lines, 1590 mm, 159.0 s at 600 mm/min",
                   abs(s_d - 0.7990) < 1e-4 and L.n_lines(30, 0.79) == 39 and (39 * 40 + 30) == 1590 and abs(1590 / 600 * 60 - 159.0) < 1e-9
                   and L.cusp_height(4, 0.79) <= 0.02))
    cut_only = sum(L.moves_length([m]) / m["f"] * 60 for m in mv if m["kind"] != "G0")
    checks.append(("the shipped boss program needs 315.5 s of cutting (G1, G3 and plunge) out of 327 s", abs(cut_only - 315.5) < 0.1))
    checks.append(("roadmap: a 2 mm cutter would leave only 0.86 mm^2 in the corners", abs(L.corner_leftover(1) - 0.86) < 0.01))
    checks.append(("roadmap: the boss area is pi * 6^2 = 113.10 mm^2", abs(math.pi * 36 - 113.10) < 0.01))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
